# Chapter 11 &mdash; Why a Recursive (or Stack-Based) Mechanism is Needed

**Concept 2 of the Chapter 11 decomposition:** *Why a Recursive (or Stack-Based) Mechanism is Needed*

REs cannot nest; an inductive English description of $L_{Dyck}$ translates directly into productions.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Why-Recursion-Is-Needed/Concept-Why-Recursion-Is-Needed.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Regular expressions have **no recursion**. Star repeats a pattern *side by side*; it
cannot **nest** one occurrence inside another. But nesting is exactly what balanced
brackets require.

The give-away is that the inductive English description translates **line for line**
into productions:

* "the empty string is balanced" &nbsp;&rarr;&nbsp; `S -> ''`
* "if $x$ is balanced then so is $(x)$" &nbsp;&rarr;&nbsp; `S -> (S)`
* "if $x$ and $y$ are balanced then so is $xy$" &nbsp;&rarr;&nbsp; `S -> SS`

A grammar is an **inductive definition written as rewriting rules**. And the machine
that recognises it needs a **stack** (Chapter 12) &mdash; unbounded memory with a
last-in-first-out discipline, which is exactly what nesting needs.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### The grammar, and an attempt to do it with a DFA

In [ ]:
Dyck = mkg({'S': ["", "(S)", "SS"]})

def depth_dfa(k):
    # a DFA that tracks depth up to k, then gives up
    lines = ['DFA', 'IF : ( -> D1', 'IF : ) -> BH']
    for d in range(1, k):
        lines += ['D%d : ( -> D%d' % (d, d+1),
                  'D%d : ) -> %s' % (d, 'IF' if d == 1 else 'D%d' % (d-1))]
    lines += ['D%d : ( -> BH' % k, 'D%d : ) -> D%d' % (k, k-1),
              'BH : ( | ) -> BH']
    return md2mc('\n'.join(lines))

## 3. Tests

The three English sentences **are** the three productions.

In [ ]:
show(Dyck)
print()
print("S -> ''    : the empty string is balanced")
print("S -> (S)   : wrap a balanced string in one pair")
print("S -> SS    : concatenate two balanced strings")

A DFA can do bounded depth &mdash; and only bounded depth.

In [ ]:
def balanced(s):
    d = 0
    for ch in s:
        d += 1 if ch == '(' else -1
        if d < 0: return False
    return d == 0

for k in [2, 3, 5]:
    D = depth_dfa(k)
    good = [s for s in language(Dyck, 2*k) if accepts_dfa(D, s)]
    toodeep = '(' * (k+1) + ')' * (k+1)
    print("depth-%d DFA: %2d states, handles %2d of the short strings, "
          "depth %d? %s" % (k, len(D["Q"]), len(good), k+1, accepts_dfa(D, toodeep)))
    assert not accepts_dfa(D, toodeep)

Every fixed $k$ fails on depth $k+1$ &mdash; and $k$ must be fixed, because $|Q|$ is finite.

In [ ]:
print("to handle depth d you need at least d+1 states")
print("Dyck has strings of EVERY depth, so no finite state count suffices.")
print()
for k in [2, 4, 8, 16]:
    print("   depth %2d -> at least %2d states" % (k, k + 1))

The grammar, by contrast, handles any depth with **three** rules.

In [ ]:
for n in [1, 5, 20, 100]:
    s = '(' * n + ')' * n
    print("  depth %-4d balanced? %s" % (n, balanced(s)))
print("\nrules in the grammar :", sum(len(v) for v in Dyck['P'].values()))
assert sum(len(v) for v in Dyck['P'].values()) == 3

What the machine needs is a **stack** &mdash; last in, first out, unbounded.

In [ ]:
def stack_check(s):
    st = []
    for ch in s:
        if ch == '(': st.append(ch)
        elif not st: return False
        else: st.pop()
    return not st

assert all(stack_check(s) == balanced(s) for s in language(Dyck, 10))
print("a stack recognises Dyck exactly -- and that is a PDA (Chapter 12).")

## 4. Animation

A depth-3 DFA: it works until the nesting outruns its states.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(depth_dfa(3), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Prove $L_{Dyck}$ is not regular with the Pumping Lemma of Chapter 4.
2. Which of the three productions makes the grammar recursive?
3. Why does nesting need LIFO rather than any unbounded counter?

In [ ]:
# Your work for the exercises above.